# 04. Mô hình hóa `ST-GAT (LSTM → GATv2)` và Đánh giá Đóng góp Các Nhóm Đặc trưng trên `PEMS08`

> Trả lời trọn vẹn **Câu hỏi Nghiên cứu 2 (RQ2)** thuần trên mạng nghiên cứu chính **`PEMS08`** bằng **PyTorch** (`SEED = 42`), giữ **đóng băng duy nhất `1` kiến trúc chuẩn** kế thừa từ văn học để tách bạch hoàn toàn tác dụng của từng nhóm đặc trưng. Tập `PEMS04` vẫn giữ độc lập hoàn toàn cho bước kiểm chứng chéo cuối cùng (`NB05`).

### Tóm tắt
* **Câu hỏi dẫn dắt (RQ2):** *Việc bổ sung các dấu hiệu tiền tắc nghẽn vào kiến trúc dự báo chú ý đồ thị tiêu chuẩn (`ST-GAT` với `GATv2`) có giúp giảm sai số dự báo (`MAE`) hay không, và từng đặc trưng / nhóm đặc trưng đóng góp ra sao?*
* **Thiết kế tối giản có căn cứ văn học:** Kế thừa khung **`ST-GAT`** (**Zhang et al., *IEEE Access* 2019** — bài báo gốc đã chứng minh vượt qua `HA / ARIMA / VAR`, `FC-LSTM` cục bộ và `STGCN` tĩnh nên không cần huấn luyện lại các baseline ngây thơ hay RNN cục bộ), nâng cấp theo cơ chế **`LSTM` nén chuỗi quá khứ `H = 12` bước trước → `2-layer Multi-Head GATv2`** (**Brody et al., *ICLR* 2022**) trên đồ thị có hướng 2 chiều (`Self / In / Out`), dự báo hồi quy thuần **`MAE` (`L1Loss`)** cho `12` bước tương lai (`5p → 60p`).
* **Kỷ luật chống rò rỉ dữ liệu:** Chia thời gian chuẩn **`60 / 20 / 20`** (**LibCity, 2021**; **BasicTS, 2025**); cả bộ chuẩn hóa `StandardScaler`, tập trạm điểm nghẽn lẫn **ngưỡng điều kiện tắc nghẽn `(v_cong, o_sat)`** đều được hiệu chuẩn **duy nhất trên tập `Train (60%)`** rồi mới áp dụng lên `Validation` và `Test`.
* **Cấu trúc 5 phần:**
  1. **Phần 1:** Thiết lập môi trường `PyTorch`, chuẩn bị tập dữ liệu `60 / 20 / 20` và kiểm toán chống rò rỉ ngưỡng tắc nghẽn thuần `Train`.
  2. **Phần 2:** Đối chứng `6` cấu hình tập đặc trưng trên cùng kiến trúc `ST-GAT (LSTM → GATv2)` theo trọng tâm **`MAE`**.
  3. **Phần 3:** Độ nhạy sai số `MAE` qua `12` tầm dự báo (`5p → 60p`) và đánh giá phụ đánh dấu điểm tắc nghẽn.
  4. **Phần 4:** Trích xuất đóng góp đặc trưng (**Permutation Feature Importance**) và phân tích trọng số chú ý cạnh động **`GATv2` ($\alpha_{ij}$)**.
  5. **Phần 5:** Tổng kết `RQ2`, kiểm thử tính quy nạp sang bản đồ `PEMS04` và bàn giao sang `NB05`.

---
## 1. Thiết lập Môi trường, Chia tập `60 / 20 / 20` và Hiệu chuẩn Ngưỡng Thuần `Train`

Đầu mục này giải đáp yêu cầu tiên quyết trước khi huấn luyện: *làm thế nào tách tập `Train / Validation / Test` theo đúng chuẩn `60 / 20 / 20` (`12 → 12` bước) với `0%` rò rỉ dữ liệu ở cả bộ chuẩn hóa, cửa sổ trượt nhân quả lẫn ngưỡng điều kiện tắc nghẽn?*

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.data import (
    ALL_FEATURE_NAMES,
    FEATURE_CONFIGS,
    FEATURE_GROUPS,
    prepare_pems_bundle,
)
from src.explain import compute_permutation_importance, extract_gatv2_edge_attention
from src.models import STGATv2, count_parameters
from src.plots_nb04 import (
    apply_paper_style,
    plot_feature_ablation_mae,
    plot_gatv2_attention_and_case,
    plot_horizon_and_congestion,
    plot_permutation_importance,
)
from src.train import set_seed, train_stgatv2_experiment

SEED = 42
set_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 4))
apply_paper_style()

CLEAN_DIR = ROOT / "datasets" / "cleaned"
RAW_DIR = ROOT / "datasets" / "data"
OUT_DIR = ROOT / "outputs"
MODEL_DIR = OUT_DIR / "models"
FIG_DIR = OUT_DIR / "figures"

bundle = prepare_pems_bundle(
    dataset_name="PEMS08",
    clean_dir=CLEAN_DIR,
    raw_dir=RAW_DIR,
    history_steps=12,
    horizon_steps=12,
    train_ratio=0.60,
    val_ratio=0.20,
    bottleneck_min_freq=0.01,
)

th_tr = bundle["train_thresholds"]
em = bundle["edge_meta"]
cg = bundle["is_congested"]
t_tr_end, t_va_end = bundle["t_train_end"], bundle["t_val_end"]
T_total, N_nodes, _ = bundle["data_raw"].shape

print(f"PyTorch: {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'} | SEED = {SEED}")
print(
    f"Ngưỡng tắc nghẽn hiệu chuẩn THUẦN trên Train (60%): "
    f"v_ff = {th_tr.v_ff:.2f} mph | v_cong = {th_tr.v_cong:.2f} mph | o_sat = {th_tr.o_sat:.4f}"
)
print(
    f"Đồ thị có hướng 2 chiều PEMS08: {em['n_nodes']} trạm, {em['n_edges_total']} cạnh "
    f"({em['n_self']} Self + {em['n_in']} In thượng nguồn + {em['n_out']} Out hạ nguồn)"
)
print(
    f"Trạm điểm nghẽn xác định thuần trên Train (tần suất >= 1%): {len(bundle['bottleneck_nodes'])} / {N_nodes} trạm "
    f"(Top 5: {bundle['bottleneck_nodes'][:5].tolist()})"
)

split_df = pd.DataFrame(
    [
        {
            "Phân đoạn (60 / 20 / 20)": "Train (60%)",
            "Khoảng chỉ số thời gian": f"[0, {t_tr_end - 1}]",
            "Số ngày (~)": round(t_tr_end / 288.0, 2),
            "Số mẫu cửa sổ (12→12)": len(bundle["t_train"]),
            "Tỷ lệ Tắc thực (%)": round(float(cg[:t_tr_end].mean() * 100.0), 2),
        },
        {
            "Phân đoạn (60 / 20 / 20)": "Validation (20%)",
            "Khoảng chỉ số thời gian": f"[{t_tr_end}, {t_va_end - 1}]",
            "Số ngày (~)": round((t_va_end - t_tr_end) / 288.0, 2),
            "Số mẫu cửa sổ (12→12)": len(bundle["t_val"]),
            "Tỷ lệ Tắc thực (%)": round(float(cg[t_tr_end:t_va_end].mean() * 100.0), 2),
        },
        {
            "Phân đoạn (60 / 20 / 20)": "Test (20%)",
            "Khoảng chỉ số thời gian": f"[{t_va_end}, {T_total - 1}]",
            "Số ngày (~)": round((T_total - t_va_end) / 288.0, 2),
            "Số mẫu cửa sổ (12→12)": len(bundle["t_test"]),
            "Tỷ lệ Tắc thực (%)": round(float(cg[t_va_end:].mean() * 100.0), 2),
        },
    ]
)
display(split_df)

> **Nhận xét (Kiểm toán chia tập `60 / 20 / 20` & Ngưỡng tắc nghẽn thuần `Train`):**
>
> * **Tính ổn định cao và `0%` rò rỉ dữ liệu:** Khi áp công thức tự tính từ Biểu đồ cơ bản **duy nhất trên `Train (60%)`** (`37,20` ngày đầu), vận tốc tự do và ngưỡng vận tốc tắc trùng khớp tuyệt đối với `NB03` ($v_{\text{ff}} = 67{,}10\text{ mph}$, $v_{\text{cong}} = 40{,}26\text{ mph}$), còn ngưỡng bão hòa $o_{\text{sat}} = 0{,}1375$ nằm gọn trong dải `11%–15%` (**Choe et al., 2002 / HCM**) và sát mức `0,1325` toàn tập. Mọi thống kê chuẩn hóa (`StandardScaler`) và danh sách `43` trạm điểm nghẽn (`>= 1%` trên `Train`) đều khóa cứng theo `Train`.
>
> * **Cách ly hoàn toàn biên cửa sổ nhân quả:** Tại mỗi phân đoạn (`Train`, `Validation`, `Test`), mẫu neo đầu tiên dịch vào `22` bước (`2H − 2` với `H = 12`) để ngay cả bước sớm nhất trong cửa sổ quá khứ `60 phút` cũng tính `7` chỉ báo tiền tắc nghẽn hoàn toàn bên trong chính phân đoạn đó — không giao cắt dữ liệu giữa các tập (`10.679` mẫu `Train`, `3.537` mẫu `Validation`, `3.538` mẫu `Test`).
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Khi giữ cố định hoàn toàn kiến trúc **`ST-GAT (LSTM → 2-layer Multi-Head GATv2)`** và chỉ thay đổi tập đặc trưng đầu vào từ `Raw (3)` qua `Base (7)`, từng nhóm riêng lẻ (`+Nhóm 1`, `+Nhóm 2`, `+Nhóm 3`) đến `Full (14)`, sai số hồi quy **`MAE`** trên tập `Test` cải thiện như thế nào?

---
## 2. Đối chứng `6` Cấu hình Tập Đặc trưng trên Kiến trúc Cố định `ST-GAT (LSTM → GATv2)`

Kế thừa khung **`ST-GAT`** (**Zhang et al., *IEEE Access* 2019** — đã chứng minh vượt `HA / ARIMA / VAR`, `FC-LSTM` cục bộ và `STGCN` tĩnh) và nâng cấp cơ chế chú ý động **`GATv2`** (**Brody et al., *ICLR* 2022**), toàn bộ `6` thí nghiệm sử dụng **cùng một kiến trúc và siêu tham số cố định**:
1. **Nén chuỗi quá khứ (`LSTM`, `hidden_dim = 32`):** Đọc trọn cửa sổ `H = 12` bước (`60 phút`) tại từng trạm để nén diễn biến vận tốc, mật độ, dao động và độ dốc vào vector trạng thái $\mathbf{h}_i(t) \in \mathbb{R}^{32}$ (vừa giữ trọn ngữ cảnh chuỗi thời gian trước khi tính chú ý đồ thị, vừa giảm chi phí tính toán đồ thị gấp `12×`).
2. **Truyền tin không gian có hướng (`2-layer Multi-Head GATv2`, `4` heads, `3` loại cạnh `Self / In / Out`):** Tính trọng số chú ý động $\alpha_{ij}(t)$ với phép phi tuyến `LeakyReLU` đặt trước tích vô hướng $\mathbf{a}^\top$, phân biệt rõ dòng tải thượng nguồn (`In`) và sóng tắc dội ngược hạ nguồn (`Out`).
3. **Đầu ra hồi quy thuần `MAE` (`L1Loss`):** Dự báo phần dư $\Delta \hat{y}$ cộng trực tiếp vào trạng thái hiện tại $[ \text{Flow}, \text{Occ}, \text{Speed} ]_t$ cho đồng thời `12` bước tương lai (`5p → 60p`), `batch_size = 128`, `epochs = 20`, `AdamW (lr = 5e-3)`, `EarlyStopping (patience = 5)`, `SEED = 42`.

*(Đặt `FORCE_RETRAIN = False` để nạp nhanh các checkpoint `SEED = 42` đã huấn luyện tại `outputs/models/stgatv2_pems08_*.pt`; đổi thành `True` để chạy lại toàn bộ `6` mô hình từ đầu).*

In [ ]:
FORCE_RETRAIN = False

TAG_MAP = {
    "Raw (3)": "raw3",
    "Base (7)": "base7",
    "Base + Nhóm 1 (10)": "base_g1",
    "Base + Nhóm 2 (9)": "base_g2",
    "Base + Nhóm 3 (9)": "base_g3",
    "Full (14)": "full14",
}

results = {}
histories = {}
summary_rows = []

for cfg_name, feat_idx in FEATURE_CONFIGS.items():
    tag = TAG_MAP[cfg_name]
    ckpt_path = MODEL_DIR / f"stgatv2_pems08_{tag}.pt"
    res = train_stgatv2_experiment(
        bundle=bundle,
        config_name=cfg_name,
        feature_indices=feat_idx,
        hidden_dim=32,
        num_lstm_layers=1,
        num_gat_layers=2,
        num_heads=4,
        dropout=0.10,
        batch_size=128,
        epochs=20,
        lr=5e-3,
        weight_decay=1e-4,
        patience=5,
        seed=SEED,
        save_path=ckpt_path,
        load_if_exists=(not FORCE_RETRAIN),
        verbose=True,
    )
    results[cfg_name] = res
    histories[cfg_name] = res["history"]
    ov = res["test_eval"]["overall"]
    summary_rows.append(
        {
            "Cấu hình": cfg_name,
            "Số kênh": len(feat_idx),
            "Tham số": res["n_params"],
            "Val L1": round(res["best_val_loss"], 4),
            "Std-MAE (3 biến)": round(ov["std_mae"], 4),
            "MAE Speed (mph)": round(ov["mae_speed"], 3),
            "RMSE Speed (mph)": round(ov["rmse_speed"], 3),
            "MAE Occ": round(ov["mae_occ"], 5),
            "MAE Flow (xe/5p)": round(ov["mae_flow"], 2),
            "MAE Speed Điểm nghẽn (mph)": round(ov["mae_speed_bn"], 3),
            "MAE Flow Điểm nghẽn": round(ov["mae_flow_bn"], 2),
        }
    )

ablation_df = pd.DataFrame(summary_rows)
raw_std = float(ablation_df.loc[ablation_df["Cấu hình"] == "Raw (3)", "Std-MAE (3 biến)"].iloc[0])
base_std = float(ablation_df.loc[ablation_df["Cấu hình"] == "Base (7)", "Std-MAE (3 biến)"].iloc[0])
ablation_df["So với Raw (%)"] = [
    f"{(v - raw_std) / raw_std * 100.0:+.2f}%" for v in ablation_df["Std-MAE (3 biến)"]
]
ablation_df["So với Base (%)"] = [
    f"{(v - base_std) / base_std * 100.0:+.2f}%" for v in ablation_df["Std-MAE (3 biến)"]
]

ablation_df.to_csv(OUT_DIR / "nb04_ablation_summary.csv", index=False)
display(ablation_df)
plot_feature_ablation_mae(ablation_df, histories, FIG_DIR / "nb04_feature_ablation_mae.png")

> **Nhận xét (Kết quả đối chứng `6` tập đặc trưng trên kiến trúc cố định `ST-GAT`):**
>
> * **Giải đáp trọn vẹn câu hỏi mở của Nhóm 4 (`STID`) từ `NB03`:** Nếu trên kiểm định tuyến tính ở `NB03`, cặp `(sin, cos)` không tự nâng `PR-AUC` do hàm tuyến tính chỉ biểu diễn được một đỉnh sóng đơn, thì trên mạng học sâu phi tuyến `ST-GAT (LSTM → GATv2)`, việc bổ sung **Nhóm 4 (`Raw 3 → Base 7`)** tạo mức giảm sai số đơn nhóm mạnh nhất: `Std-MAE` giảm **`-6,15%`** (`0,1771 → 0,1662`), `MAE Flow` giảm **`-8,07%`** (`19,21 → 17,66 xe/5p`), `MAE Occ` giảm **`-6,44%`** (`0,00777 → 0,00727`), và `MAE Speed` giảm **`-4,90%`** (`1,469 → 1,397 mph`).
>
> * **Thứ hạng độc lập của `3` nhóm tín hiệu tiền tắc nghẽn trùng khớp `100%` với `NB03`:** Đặt riêng từng nhóm lên nền `Base (7)` đều giảm đồng thời cả `Val L1` lẫn `Test Std-MAE` theo đúng thứ tự đã phát hiện ở RQ1: **Nhóm 1 Dao động khử dốc mạnh nhất** (`Val L1 = 0,1675`, `Std-MAE = 0,1652`, `-0,60%` vs `Base`), kế đến là **Nhóm 2 Liên kết đa biến** (`0,1682`, `0,1656`, `-0,36%`, giảm mạnh `Flow` xuống `17,48`), và **Nhóm 3 Độ dốc có dấu** (`0,1685`, `0,1659`, `-0,18%`).
>
> * **Hiệu ứng cộng hưởng vượt tổng từng nhóm lẻ khi gộp đủ `14` đặc trưng (`Full 14`):** Khi đưa đồng thời cả `7` chỉ báo tiền tắc nghẽn vào `Base (7)`, `ST-GAT Full (14)` đạt kết quả tốt nhất trên **mọi thước đo hồi quy chính** (`Val L1 = 0,1647`, `Test Std-MAE = 0,1626`, `MAE Speed = 1,371 mph`, `MAE Occ = 0,00712`, `MAE Flow = 17,20 xe/5p`, `MAE Speed Điểm nghẽn = 2,550 mph`). Đáng chú ý, mức giảm `Std-MAE` so với `Base (7)` đạt **`-0,0036` (`-2,17%`)** — lớn gần **gấp đôi** tổng mức giảm cộng gộp của ba nhóm chạy lẻ (`0,0010 + 0,0006 + 0,0003 = 0,0019`), và giảm tổng cộng **`-8,19%`** so với `Raw (3)` (`MAE Flow -10,46%`, `MAE Occ -8,37%`, `MAE Speed -6,67%`).
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Mức cải thiện `MAE` của `Full (14)` so với `Raw (3)` và `Base (7)` thay đổi ra sao khi tầm dự báo kéo dài từ `5 phút` tới `60 phút`, và khi áp điều kiện tắc nghẽn thuần `Train` làm đánh giá phụ lên kết quả dự báo thì chất lượng đánh dấu điểm tắc có được nâng lên hay không?

---
## 3. Phân tích Độ nhạy Theo Tầm Dự báo (`5p → 60p`) và Đánh giá Phụ Đánh dấu Tắc nghẽn

Phần này tách chi tiết sai số `MAE` tại các mốc chuẩn văn học (**`15 phút`**, **`30 phút`**, **`60 phút`** — theo **ST-GAT, 2019** & **BasicTS, 2025**) cùng đường cong đầy đủ `12` bước tương lai, đồng thời thực hiện **đánh giá phụ gọn nhẹ**: áp trực tiếp quy tắc vật lý đã hiệu chuẩn thuần trên `Train (60%)` ($v < 40{,}26\text{ mph} \land o \ge 0{,}1375$) lên chuỗi dự báo $(\hat{\text{Speed}}, \hat{\text{Occupancy}})$ mà không thêm bất kỳ nhánh phân loại phụ trợ nào.

In [ ]:
horizon_rows = []
for h_min in [15, 30, 60]:
    for cfg_name in ["Raw (3)", "Base (7)", "Base + Nhóm 1 (10)", "Base + Nhóm 2 (9)", "Base + Nhóm 3 (9)", "Full (14)"]:
        hm = results[cfg_name]["test_eval"]["by_horizon"][h_min]
        horizon_rows.append(
            {
                "Tầm dự báo": f"{h_min} phút",
                "Cấu hình": cfg_name,
                "Std-MAE": round(hm["std_mae"], 4),
                "MAE Speed (mph)": round(hm["mae_speed"], 3),
                "MAE Occ": round(hm["mae_occ"], 5),
                "MAE Flow (xe/5p)": round(hm["mae_flow"], 2),
                "MAE Speed Điểm nghẽn": round(hm["mae_speed_bn"], 3),
                "F1 Tắc (phụ)": round(hm["cong_f1"], 4),
                "Precision (phụ)": round(hm["cong_prec"], 4),
                "Recall (phụ)": round(hm["cong_rec"], 4),
            }
        )

horizon_df = pd.DataFrame(horizon_rows)
horizon_df.to_csv(OUT_DIR / "nb04_horizon_results.csv", index=False)
display(horizon_df)

# Full 12-step MAE curves (5..60 min) for Raw(3), Base(7), Full(14)
horizon_curves = {}
for cfg_name in ["Raw (3)", "Base (7)", "Full (14)"]:
    yt = results[cfg_name]["test_eval"]["y_true_raw"]
    yp = results[cfg_name]["test_eval"]["y_pred_raw"]
    ae = np.abs(yp - yt)
    horizon_curves[cfg_name] = {
        "mae_speed": ae[..., 2].mean(axis=(0, 1)),
        "mae_occ": ae[..., 1].mean(axis=(0, 1)),
        "mae_flow": ae[..., 0].mean(axis=(0, 1)),
    }

# Compact secondary congestion-marking comparison table
cong_rows = []
for h_min in [15, 30, 60]:
    row = {"Tầm dự báo": f"{h_min} phút"}
    for cfg_name in ["Raw (3)", "Base (7)", "Full (14)"]:
        hm = results[cfg_name]["test_eval"]["by_horizon"][h_min]
        row[f"F1 {cfg_name}"] = round(hm["cong_f1"], 4)
        row[f"Prec {cfg_name}"] = round(hm["cong_prec"], 4)
        row[f"Rec {cfg_name}"] = round(hm["cong_rec"], 4)
    cong_rows.append(row)
cong_check_df = pd.DataFrame(cong_rows)

ov_raw = results["Raw (3)"]["test_eval"]["overall"]
ov_base = results["Base (7)"]["test_eval"]["overall"]
ov_full = results["Full (14)"]["test_eval"]["overall"]
print(
    f"Tổng hợp đánh giá phụ (Avg 5–60p, ngưỡng thuần Train): "
    f"F1: Raw={ov_raw['cong_f1']:.4f} -> Base={ov_base['cong_f1']:.4f} -> Full={ov_full['cong_f1']:.4f} | "
    f"Precision: Raw={ov_raw['cong_prec']:.4f} -> Base={ov_base['cong_prec']:.4f} -> Full={ov_full['cong_prec']:.4f}"
)

plot_horizon_and_congestion(
    horizon_curves,
    cong_check_df,
    FIG_DIR / "nb04_horizon_and_congestion_check.png",
)

> **Nhận xét (Mở rộng lợi thế ở tầm xa `30–60 phút` & Kiểm chứng phụ đánh dấu tắc nghẽn):**
>
> * **Lợi ích của `Full (14)` tăng rõ rệt khi tầm dự báo kéo dài (Hình 2A–B):**
>
>   * Tại **`15 phút`**, `Full (14)` giảm `MAE Speed` từ `1,173` (`Raw`) / `1,152` (`Base`) xuống **`1,141 mph`** và `MAE Flow` từ `16,55` / `15,90` xuống **`15,62 xe/5p`**.
>
>   * Tại **`30 phút`**, khoảng cách mở rộng: `MAE Speed` giảm từ `1,494` / `1,428` xuống **`1,401 mph`**, `MAE Flow` từ `18,83` / `17,55` xuống **`17,02 xe/5p`** (`Std-MAE` giảm `-7,45%` vs `Raw`, `-2,21%` vs `Base`).
>
>   * Tại **`60 phút`** (khó nhất), **`Full (14)` tạo bước nhảy lớn nhất**: `MAE Speed` giảm từ `1,911` (`Raw`) và `1,775` (`Base`) xuống **`1,729 mph`** (**`-9,52%`** vs `Raw`, **`-2,59%`** vs `Base`); `MAE Flow` giảm từ `24,01` và `20,81` xuống **`20,17 xe/5p`** (**`-15,99%`** vs `Raw`, **`-3,08%`** vs `Base`); `MAE Occ` giảm từ `0,00975` và `0,00872` xuống **`0,00849`** (**`-12,92%`** vs `Raw`, **`-2,64%`** vs `Base`).
>
> * **Đánh giá phụ đánh dấu điểm tắc nghẽn (Hình 2C — ngưỡng hiệu chuẩn thuần `Train`):** Không cần thêm nhánh phân loại, riêng đầu ra hồi quy của `Full (14)` giữ `F1` trung bình cao nhất (`0,6923` vs `0,6911` `Base` và `0,6866` `Raw`), đồng thời nâng `Precision` trên cả `3` mốc (trung bình `0,7812 → 0,8015 → 0,8166`, tăng **`+3,54%pt`** so với `Raw` nhờ giảm báo động giả). Đặc biệt tại mốc xa **`60 phút`**, `Full (14)` vượt trội đồng thời cả `F1` (**`0,5511`** vs `0,5381` `Base` và `0,5280` `Raw`), `Precision` (**`0,7427`**) lẫn `Recall` (**`0,4381`**). *(Ghi nhận trung thực: ở mốc ngắn `15–30p`, `F1` giữa `Base` và `Full` gần tương đương nhau quanh `0,78` và `0,68` do hàm mất mát hồi quy `L1` tối ưu đều cho toàn mạng 98% thông thoáng thay vì gán trọng số riêng cho vùng kẹt cực đoan).*
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Bên trong mô hình `ST-GAT Full (14)`, cụ thể **những đặc trưng nào** trong `14` đầu vào được mô hình dựa vào nhiều nhất để giảm `MAE`, và cơ chế chú ý động **`GATv2` ($\alpha_{ij}$)** phân bổ trọng số giữa `Self`, `In` (thượng nguồn) và `Out` (hạ nguồn dội ngược) ra sao quanh các điểm nghẽn?

---
## 4. Trích xuất Đóng góp Đặc trưng (`Permutation Feature Importance`) và Chú ý Cạnh `GATv2`

Để trả lời vế thứ hai của RQ2 (*từng đặc trưng và nhóm đặc trưng đóng góp ra sao?*), ta thực hiện hai phép phân tích trực tiếp trên mô hình **`ST-GAT Full (14)`** trên tập `Test` (không cần huấn luyện lại):
1. **Permutation Feature Importance (`5` lần lặp, `1.024` mẫu `Test`, `SEED = 42`):** Xáo trộn ngẫu nhiên từng đặc trưng (trong `14` biến) và từng nhóm (trong `5` nhóm) dọc theo trục thời gian để đo trực tiếp mức tăng sai số **`ΔStd-MAE`**, **`ΔMAE Speed`**, **`ΔMAE Occ`**, và **`ΔMAE Flow`**. Biến nào khi bị phá cấu trúc làm `MAE` tăng càng mạnh thì càng đóng vai trò quyết định.
2. **Trọng số chú ý cạnh động `GATv2` ($\alpha_{ij}$):** Tách bạch tổng chú ý phân bổ cho **`Self`**, **`In (Thượng nguồn)`** và **`Out (Hạ nguồn dội ngược)`** giữa trạng thái bình thường và trạng thái tiền tắc nghẽn (`t + 15p` bị tắc theo ngưỡng thuần `Train`) trên các trạm điểm nghẽn và chi tiết tại điểm nghẽn số 1 **`N22`**.

In [ ]:
full_model = results["Full (14)"]["model"]

pfi_res = compute_permutation_importance(
    model=full_model,
    bundle=bundle,
    feature_indices=FEATURE_CONFIGS["Full (14)"],
    n_repeats=5,
    max_samples=1024,
    batch_size=256,
    seed=SEED,
)

group_pfi_df = pfi_res["group_df"]
feat_pfi_df = pfi_res["feature_df"]
feat_pfi_df.to_csv(OUT_DIR / "nb04_permutation_importance.csv", index=False)

print("Bảng 4.1 — Tầm quan trọng theo Nhóm đặc trưng (Test PFI, 5 lần lặp):")
display(
    group_pfi_df[
        ["group", "n_features", "delta_std_mae", "delta_std_mae_sd", "delta_mae_speed", "delta_mae_occ", "delta_mae_flow"]
    ].rename(
        columns={
            "group": "Nhóm đặc trưng",
            "n_features": "Số biến",
            "delta_std_mae": "ΔStd-MAE",
            "delta_std_mae_sd": "±SD",
            "delta_mae_speed": "ΔMAE Speed",
            "delta_mae_occ": "ΔMAE Occ",
            "delta_mae_flow": "ΔMAE Flow",
        }
    ).round(5)
)

print("Bảng 4.2 — Xếp hạng toàn bộ 14 đặc trưng riêng lẻ trên ST-GAT Full (14):")
display(
    feat_pfi_df[
        ["feature", "group", "delta_std_mae", "delta_std_mae_sd", "delta_mae_speed", "delta_mae_occ", "delta_mae_flow"]
    ].rename(
        columns={
            "feature": "Đặc trưng",
            "group": "Nhóm",
            "delta_std_mae": "ΔStd-MAE",
            "delta_std_mae_sd": "±SD",
            "delta_mae_speed": "ΔMAE Speed",
            "delta_mae_occ": "ΔMAE Occ",
            "delta_mae_flow": "ΔMAE Flow",
        }
    ).round(5)
)

plot_permutation_importance(group_pfi_df, feat_pfi_df, FIG_DIR / "nb04_permutation_importance.png")

In [ ]:
attn_res = extract_gatv2_edge_attention(
    model=full_model,
    bundle=bundle,
    feature_indices=FEATURE_CONFIGS["Full (14)"],
    target_horizon_step=3,
    top_node=None,
    batch_size=256,
)

sm = attn_res["summary_means"]
attn_agg_df = pd.DataFrame(
    [
        {
            "Hướng cạnh GATv2": "Self (Bản thân)",
            "Bình thường (∑α_ij)": round(sm["self_normal"], 4),
            "Tiền tắc nghẽn (∑α_ij)": round(sm["self_pre_cong"], 4),
            "Dịch chuyển (Δα)": round(sm["self_pre_cong"] - sm["self_normal"], 4),
        },
        {
            "Hướng cạnh GATv2": "In (Thượng nguồn đổ vào)",
            "Bình thường (∑α_ij)": round(sm["in_normal"], 4),
            "Tiền tắc nghẽn (∑α_ij)": round(sm["in_pre_cong"], 4),
            "Dịch chuyển (Δα)": round(sm["in_pre_cong"] - sm["in_normal"], 4),
        },
        {
            "Hướng cạnh GATv2": "Out (Hạ nguồn dội ngược)",
            "Bình thường (∑α_ij)": round(sm["out_normal"], 4),
            "Tiền tắc nghẽn (∑α_ij)": round(sm["out_pre_cong"], 4),
            "Dịch chuyển (Δα)": round(sm["out_pre_cong"] - sm["out_normal"], 4),
        },
    ]
)

print(f"Tổng hợp trọng số chú ý GATv2 trên {len(attn_res['bn_summary_df'])} trạm điểm nghẽn có đủ 2 trạng thái trên Test:")
display(attn_agg_df)

print(f"Chi tiết các cạnh truyền tin vào điểm nghẽn số 1 — N{attn_res['top_node']}:")
top_ed = attn_res["top_edges_df"].copy()
out_norm_top = float(top_ed.loc[top_ed["edge_type"] == 2, "alpha_normal"].sum())
out_cong_top = float(top_ed.loc[top_ed["edge_type"] == 2, "alpha_pre_cong"].sum())
print(
    f"Tổng chú ý chiều Out (Hạ nguồn dội ngược) tại N{attn_res['top_node']}: "
    f"{out_norm_top:.4f} (Bình thường) -> {out_cong_top:.4f} (Tiền tắc nghẽn, {out_cong_top - out_norm_top:+.4f})"
)
display(top_ed[["src", "dst", "direction", "alpha_normal", "alpha_pre_cong", "alpha_shift"]].round(4))

plot_gatv2_attention_and_case(
    attn_res=attn_res,
    y_true_test=results["Full (14)"]["test_eval"]["y_true_raw"],
    y_pred_base=results["Base (7)"]["test_eval"]["y_pred_raw"],
    y_pred_full=results["Full (14)"]["test_eval"]["y_pred_raw"],
    t_test=bundle["t_test"],
    train_thresholds=bundle["train_thresholds"],
    horizon_step=3,
    save_path=FIG_DIR / "nb04_gatv2_attention_and_case.png",
)

> **Nhận xét (Xếp hạng đóng góp `14` đặc trưng & Cơ chế chú ý động `GATv2`):**
>
> * **Toàn bộ `14 / 14` đặc trưng đều có đóng góp dương có ý nghĩa (`ΔStd-MAE > 0`, không có biến thừa — Hình 3):** Ngoài bộ `3` biến đích gốc (`Flow +0,2446`, `Occupancy +0,2293`, `Speed +0,1477`), thứ hạng đóng góp của `4` nhóm đặc trưng bổ sung trong `ST-GAT Full (14)` cho thấy cơ chế bổ trợ rất rõ ràng:
>
>   1. **Nhóm 4 — `STID` (`ΔStd-MAE = +0,0686 ± 0,0018`, `ΔSpeed = +0,291 mph`, `ΔFlow = +12,31`):** Đứng đầu trong `4` nhóm bổ sung nhờ cặp chu kỳ 24 giờ (`tod_cos +0,0567`, `tod_sin +0,0342`) và chu kỳ tuần (`dow_sin +0,0055`, `dow_cos +0,0012`), giải thích chính xác bước nhảy lớn `Raw (3) → Base (7)`.
>
>   2. **Nhóm 3 — Độ dốc có dấu (`ΔStd-MAE = +0,0248 ± 0,0007`, `ΔSpeed = +0,120 mph`, `ΔFlow = +4,51`):** Trở thành **nhóm quan trọng nhất trong `7` tín hiệu tiền tắc nghẽn** khi đặt trong tổ hợp `Full (14)`, dẫn đầu bởi **`occ_slope` (`+0,0222`)** và **`spd_slope` (`+0,0067`, đóng góp riêng `+0,087 mph` cho `MAE Speed`)**. Khi đứng một mình (`Base + Nhóm 3`), độ dốc chưa đủ tách dao động nhiễu; nhưng khi phối hợp với Nhóm 1–2 trong `Full (14)`, cặp độ dốc có dấu trở thành kim chỉ nam định hướng tăng/giảm cho dự báo đa bước — giải thích trực tiếp hiệu ứng cộng hưởng gấp đôi ở Phần 2.
>
>   3. **Nhóm 1 — Dao động khử dốc (`ΔStd-MAE = +0,0051 ± 0,0001`, `ΔSpeed = +0,038 mph`):** Dẫn đầu bởi **`occ_rstd` (`+0,0042`)**, tiếp đến `flow_rstd` (`+0,0023`) và `spd_rstd` (`+0,0018`).
>
>   4. **Nhóm 2 — Liên kết đa biến (`ΔStd-MAE = +0,0042 ± 0,0001`, `ΔSpeed = +0,032 mph`):** **`flow_occ_decouple` (`+0,0032`)** — tín hiệu đứt gãy Biểu đồ cơ bản — đóng góp gấp `2,4×` so với `occ_spd_corr` (`+0,0013`).
>
> * **Trọng số chú ý động `GATv2` khớp chặt vật lý sóng tắc dội ngược từ hạ nguồn (Hình 4A–B):** Trên trung bình `40` trạm điểm nghẽn có đủ 2 trạng thái trên `Test`, mô hình phân bổ `57%–66%` trọng số chú ý cho các nhánh lân cận; khi chuyển sang trạng thái tiền tắc nghẽn, chú ý cho chiều **`Out` (hạ nguồn dội ngược)** tăng từ `0,2401 → 0,2511` (`+0,0110`) và `Self` tăng `0,3385 → 0,4296`, trong khi chiều `In` giảm `0,4214 → 0,3193`. Đặc biệt tại **điểm nghẽn số 1 `N22`**, toàn bộ **`4 / 4` cạnh hạ nguồn `Out`** đều tăng mạnh chú ý trước khi tắc (`N149 → N22: +0,0588`, `N30 → N22: +0,0496`, `N52 → N22: +0,0402`, `N4 → N22: +0,0098`), kéo **tổng trọng số chiều `Out` tại `N22` tăng vọt từ `0,2376` (`23,8%`) lên `0,3961` (`39,6%`, tăng `+66,7%` tương đối)** — nhất quán trực tiếp với cơ chế lan truyền ngược dòng đã kiểm định ở `NB03` Phần 4 (**DCRNN, 2018**; **PDFormer, 2023**).
>
> **Câu hỏi nghi vấn tiếp theo:**
>
> Mô hình **`ST-GAT (LSTM → GATv2)`** vừa huấn luyện trên `PEMS08` (`170` trạm) có thật sự quy nạp (*inductive*) để nạp trực tiếp lên bản đồ **`PEMS04` (`307` trạm)** không, và những đầu ra nào cần bàn giao sang bước kiểm chứng chéo độc lập ở `NB05`?

---
## 5. Tổng kết Câu hỏi Nghiên cứu 2 & Kiểm thử Tính Quy nạp Bàn giao sang `NB05`

Kiểm tra trực tiếp khả năng chuyển đổi cấu trúc đồ thị (*inductive graph swap*) của `ST-GAT Full (14)` từ `PEMS08` (`170` trạm, `718` cạnh) sang đồ thị `PEMS04` (`307` trạm) mà không thay đổi bất kỳ trọng số nào, nhằm đảm bảo toàn bộ pipeline `PyTorch` sẵn sàng cho `NB05`.

In [ ]:
# Inductive compatibility verification on PEMS04 graph topology (shape & attention dry-run only; full evaluation belongs to NB05)
pems04_edges_df = pd.read_csv(RAW_DIR / "PEMS04" / "PEMS04.csv")
pems04_edges_df.columns = [c.strip().lower() for c in pems04_edges_df.columns]
from src.data import build_directed_edge_index

ei_04, et_04, meta_04 = build_directed_edge_index(pems04_edges_df, num_nodes=307)
test_model = STGATv2(
    in_channels=14,
    edge_index=bundle["edge_index"],
    edge_type=bundle["edge_type"],
    hidden_dim=32,
    num_lstm_layers=1,
    num_gat_layers=2,
    num_heads=4,
    horizon_steps=12,
    out_vars=3,
    dropout=0.10,
)
test_model.load_state_dict(torch.load(MODEL_DIR / "stgatv2_pems08_full14.pt", map_location="cpu", weights_only=False)["state_dict"])
test_model.eval()
test_model.set_graph(ei_04, et_04)

with torch.no_grad():
    dummy_04 = torch.zeros((2, 12, 307, 14), dtype=torch.float32)
    y_04, a_04 = test_model.forward_with_attention(dummy_04)

print(
    f"Kiểm thử quy nạp sang PEMS04 ({meta_04['n_nodes']} trạm, {meta_04['n_edges_total']} cạnh = "
    f"{meta_04['n_self']} Self + {meta_04['n_in']} In + {meta_04['n_out']} Out): "
    f"output = {tuple(y_04.shape)}, attention = {tuple(a_04.shape)} — HỢP LỆ 100%."
)

> **Nhận xét (Tổng kết trọn vẹn RQ2 trên `PEMS08` & Bàn giao sang `NB05`):**
>
> 1. **Kỷ luật chống rò rỉ tuyệt đối:** Chia thời gian chuẩn `60 / 20 / 20` (`12 → 12` bước), hiệu chuẩn `StandardScaler`, `43` trạm điểm nghẽn và ngưỡng tắc nghẽn ($v_{\text{cong}} = 40{,}26\text{ mph}$, $o_{\text{sat}} = 0{,}1375$) **thuần trên `Train (60%)`**.
>
> 2. **Bằng chứng thực nghiệm trả lời RQ2 trên kiến trúc đóng băng `ST-GAT (LSTM → 2-layer GATv2)` (`11.524 – 12.932` tham số):**
>    * Bổ sung **Nhóm 4 `STID` (`Raw 3 → Base 7`)** giảm `Test Std-MAE` **`-6,15%`** (`0,1771 → 0,1662`), chứng minh giá trị mạnh của chu kỳ thời gian trong mô hình học sâu phi tuyến.
>    * Bổ sung riêng từng nhóm tín hiệu tiền tắc nghẽn lên `Base (7)` đều giảm `MAE` theo đúng thứ tự `NB03`: **Nhóm 1 Dao động (`0,1652`) > Nhóm 2 Liên kết (`0,1656`) > Nhóm 3 Độ dốc (`0,1659`)**.
>    * Gộp trọn bộ **`Full (14)`** tạo hiệu ứng cộng hưởng vượt tổng từng nhóm lẻ: **`Test Std-MAE = 0,1626`** (**`-8,19%`** so với `Raw 3`, **`-2,17%`** so với `Base 7`), `MAE Speed = 1,371 mph`, `MAE Occ = 0,00712`, `MAE Flow = 17,20 xe/5p`, cải thiện rõ nhất ở tầm xa **`60 phút`** (`MAE Speed -9,52%` vs `Raw`, `-2,59%` vs `Base`; `F1` đánh dấu tắc phụ tăng lên **`0,5511`**).
>
> 3. **Trích xuất đóng góp & Giải thích vật lý:** Toàn bộ `14 / 14` đặc trưng đều có `Permutation Importance > 0` (`Nhóm 4 STID +0,0686` > `Nhóm 3 Độ dốc +0,0248` với `occ_slope / spd_slope` > `Nhóm 1 Dao động +0,0051` với `occ_rstd` > `Nhóm 2 Liên kết +0,0042` với `flow_occ_decouple`); `GATv2` tự động dồn mạnh trọng số chú ý sang các cạnh hạ nguồn dội ngược (`Out`) trước khi tắc (tại `N22` tăng từ `23,8% → 39,6%`).
>
> 4. **Đầu ra bàn giao:** `6` checkpoint PyTorch (`outputs/models/stgatv2_pems08_*.pt`), `3` bảng tổng hợp (`outputs/nb04_ablation_summary.csv`, `outputs/nb04_horizon_results.csv`, `outputs/nb04_permutation_importance.csv`), và `4` biểu đồ `300 DPI` (`outputs/figures/nb04_*.png`).
>
> **Câu hỏi nghi vấn chuyển sang Bước 05 (Kiểm chứng độc lập trên `PEMS04`):**
>
> Khi mang trọn bộ quy trình tự hiệu chuẩn ngưỡng (thuần trên `Train 60%` của `PEMS04`), bộ `14` đặc trưng và kiến trúc quy nạp **`ST-GAT (LSTM → GATv2)`** sang bản đồ độc lập **`PEMS04` (`307` trạm, khu vực vịnh San Francisco — chưa từng chạm tới trong `NB03–NB04`)**, hiệu quả giảm `MAE` của `7` chỉ báo tiền tắc nghẽn và thứ hạng đóng góp đặc trưng có còn lặp lại nhất quán hay không?
>
> **Tiếp theo:** `05_Cross_Map_Validation_PEMS04.ipynb` *(Kế hoạch Bước 05)*